In [1]:
import os
from glob import glob
from IPython.display import HTML, display
from pathlib import Path

In [2]:
vevo_orig_list_path = "/home/lxj220018/GenVC-EMO/Emotion_Transfer/data/vevo_test_out/any2any_same_all"
vevo_emo_list_path = "/home/lxj220018/GenVC-EMO/Amphion/output_esd/All_test_deep_prefix(perceiver(gst)+emo)_epoch-0012_step-0047000_loss-0.115780"

## get the list of files in the vevo_orig_list_path
vevo_orig_list = glob(os.path.join(vevo_orig_list_path, "*.wav"))
vevo_emo_list = glob(os.path.join(vevo_emo_list_path, "*.wav"))

target_spk = "0011"
target_base_range = [320,350]

def get_base_id(utt):
    utt_id = int(utt.split("/")[-1].split(".")[0].split("_")[1])
    neu_id = utt_id % 350
    if neu_id == 0:
        neu_id = 350
    return neu_id


In [3]:
for vevo_orig_utt in vevo_emo_list:
    base_id = get_base_id(vevo_orig_utt)
    if base_id >= target_base_range[0] and base_id <= target_base_range[1]:
        print(vevo_orig_utt)
        break



/home/lxj220018/GenVC-EMO/Amphion/output_esd/All_test_deep_prefix(perceiver(gst)+emo)_epoch-0012_step-0047000_loss-0.115780/0011_001036_Happy_to_0011_000216_Neutral.wav


In [4]:
proposed_demo_path = "/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780"
wavs = glob(os.path.join(proposed_demo_path, "*.wav"))

In [6]:
def get_reference_wav(wav_path):
    esd_path = "/home/lxj220018/GenVC-EMO/ESD/"
    base_name = wav_path.split("/")[-1].split(".")[0]
    parts = base_name.split("_to_")
    spk, utt_id, emo = parts[1].split("_")
    reference_wav = os.path.join(esd_path, f"{spk}/{emo}/{spk}_{utt_id}.wav")
    assert os.path.exists(reference_wav)
    return reference_wav

def get_source_wav(wav_path):
    esd_path = "/home/lxj220018/GenVC-EMO/ESD/"

    base_name = wav_path.split("/")[-1].split(".")[0]
    parts = base_name.split("_to_")
    spk, utt_id, emo = parts[0].split("_")
    utt_id = int(utt_id)
    src_wav = os.path.join(esd_path, f"{spk}/{emo}/{spk}_{utt_id:06d}.wav")
    assert os.path.exists(src_wav)
    return src_wav

def get_gt_wav(wav_path):
    esd_path = "/home/lxj220018/GenVC-EMO/ESD/"
    neu_start = 0
    ang_start = 350
    hap_start = 700
    sad_start = 1050
    sur_start = 1400

    base_name = wav_path.split("/")[-1].split(".")[0]
    parts = base_name.split("_to_")
    spk, utt_id, emo = parts[0].split("_")
    _, _, target_emo = parts[1].split("_")
    root_id = int(utt_id) % 350
    if root_id == 0:
        root_id = 350
    if target_emo == "Angry":
        root_id += ang_start
    if target_emo == "Happy":
        root_id += hap_start
    if target_emo == "Sad":
        root_id += sad_start
    if target_emo == "Surprise":
        root_id += sur_start
    if target_emo == "Neutral":
        root_id = root_id
    # print(os.path.join(esd_path, f"{spk}/{target_emo}/{spk}_{root_id:06d}.wav"))
    gt_wav = os.path.join(esd_path, f"{spk}/{target_emo}/{spk}_{root_id:06d}.wav")
    assert os.path.exists(gt_wav)
    return gt_wav
    
def get_baseline_wav(wav_path):
    base_line_path = "/home/lxj220018/GenVC-EMO/Amphion/output_demo"
    base_name = wav_path.split("/")[-1].split(".")[0]
    gt_name = base_name + "_vevovoice.wav"
    gt_wav = os.path.join(base_line_path, "vevovoice", gt_name)
    # print(gt_wav)
    assert os.path.exists(gt_wav)
    return gt_wav

In [23]:
import shutil

chosen_list = "./demo_chosen_list_half.txt"
root_dir = "/home/lxj220018/GenVC-EMO/Amphion/output_demo/"

sample_gt_dir = "./Samples/gt/"
sample_bl_dir = "./Samples/baseline/"
sample_pp_dir = "./Samples/proposed/"

os.makedirs(sample_gt_dir, exist_ok=True)
os.makedirs(sample_bl_dir, exist_ok=True)
os.makedirs(sample_pp_dir, exist_ok=True)

direction_to_wavs = {}
current_direction = None

with open(chosen_list, "r", encoding="utf-8") as f:
    for raw in f:
        line = raw.strip()
        if not line:
            continue

        # Header line, e.g. "Neutral --> Angry"
        if line.startswith('"') and "-->" in line:
            current_direction = line.split('"')[1].split("##")[0].strip()
            direction_to_wavs[current_direction] = []
            continue

        # Wav path line
        if current_direction is not None and line.endswith(".wav"):
            proposed_wav = line.replace("./", root_dir)
            if not os.path.isfile(proposed_wav):
                continue

            source_wav = get_source_wav(proposed_wav)
            baseline_wav = get_baseline_wav(proposed_wav)

            source_name = os.path.basename(source_wav)
            baseline_name = os.path.basename(baseline_wav)
            proposed_name = os.path.basename(proposed_wav)

            shutil.copy2(source_wav, os.path.join(sample_gt_dir, source_name))
            shutil.copy2(baseline_wav, os.path.join(sample_bl_dir, baseline_name))
            shutil.copy2(proposed_wav, os.path.join(sample_pp_dir, proposed_name))

            direction_to_wavs[current_direction].append(
                {
                    "direction": current_direction.replace(" --> ", " -> "),
                    "source": f"./Samples/gt/{source_name}",
                    "baseline": f"./Samples/baseline/{baseline_name}",
                    "proposed": f"./Samples/proposed/{proposed_name}",
                }
            )

# # keep only first 2 samples per direction
# for k in direction_to_wavs:
#     direction_to_wavs[k] = direction_to_wavs[k][:2]

direction_to_wavs

{'Neutral --> Angry': [{'direction': 'Neutral -> Angry',
   'source': './Samples/gt/0011_000326.wav',
   'baseline': './Samples/baseline/0011_000326_Neutral_to_0011_000661_Angry_vevovoice.wav',
   'proposed': './Samples/proposed/0011_000326_Neutral_to_0011_000661_Angry.wav'},
  {'direction': 'Neutral -> Angry',
   'source': './Samples/gt/0019_000346.wav',
   'baseline': './Samples/baseline/0019_000346_Neutral_to_0019_000663_Angry_vevovoice.wav',
   'proposed': './Samples/proposed/0019_000346_Neutral_to_0019_000663_Angry.wav'}],
 'Neutral --> Happy': [{'direction': 'Neutral -> Happy',
   'source': './Samples/gt/0019_000350.wav',
   'baseline': './Samples/baseline/0019_000350_Neutral_to_0019_001017_Happy_vevovoice.wav',
   'proposed': './Samples/proposed/0019_000350_Neutral_to_0019_001017_Happy.wav'},
  {'direction': 'Neutral -> Happy',
   'source': './Samples/gt/0011_000327.wav',
   'baseline': './Samples/baseline/0011_000327_Neutral_to_0011_001018_Happy_vevovoice.wav',
   'proposed': '

In [25]:
import json

with open("chosen_samples.json", "w", encoding="utf-8") as f:
    json.dump(direction_to_wavs, f, indent=2, ensure_ascii=False)

In [22]:
## Let's group the wavs by the source and target emotion
wav_group = {}
for wav in wavs:
    wav_name = wav.split("/")[-1].split(".")[0]
    parts = wav_name.split("_to_")
    source_emo = parts[0].split("_")[-1]
    target_emo = parts[1].split("_")[-1]
    key = f"{source_emo} --> {target_emo}"
    if key not in wav_group:
        wav_group[key] = []
    wav_group[key].append(wav)

for key, wavs in wav_group.items():
    print(key, len(wavs))
    # break


Sad --> Happy 16


In [8]:
wav_group

{'Angry --> Happy': ['/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0013_000677_Angry_to_0013_001016_Happy.wav',
  '/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_000683_Angry_to_0019_001018_Happy.wav',
  '/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0016_000698_Angry_to_0016_001010_Happy.wav',
  '/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0011_000684_Angry_to_0011_001000_Happy.wav',
  '/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0013_000687_Angry_to_0013_001003_Happy.wav',
  '/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_000693_Angry_to_0019_001007_Happy.wav',
  '/home/lxj220018/GenVC-EMO/Amphion/output_demo/demo_tes

In [22]:
def audio_tag(path):
    p = Path(path)
    return f"""
    <audio controls preload="none" style="width:260px">
      <source src="{p.as_posix()}" type="audio/wav">
    </audio>
    """

def show_triplet(case_name, reference, gt, baseline, proposed, reference_name="Reference"):
    if reference_name == "Reference":
      html = f"""
      <table style="border-collapse:collapse;">
        <tr>
          <th style="padding:6px;border:1px solid #ddd;">Case</th>
          <th style="padding:6px;border:1px solid #ddd;">Reference</th>
          <th style="padding:6px;border:1px solid #ddd;">Baseline</th>
          <th style="padding:6px;border:1px solid #ddd;">Proposed</th>
        </tr>
        <tr>
          <td style="padding:6px;border:1px solid #ddd;">{case_name}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(reference)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(baseline)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(proposed)}</td>
        </tr>
      </table>
      """
    if reference_name == "GT":
      html = f"""
      <table style="border-collapse:collapse;">
        <tr>
          <th style="padding:6px;border:1px solid #ddd;">Case</th>
          <th style="padding:6px;border:1px solid #ddd;">GT</th>
          <th style="padding:6px;border:1px solid #ddd;">Baseline</th>
          <th style="padding:6px;border:1px solid #ddd;">Proposed</th>
        </tr>
        <tr>
          <td style="padding:6px;border:1px solid #ddd;">{case_name}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(gt)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(baseline)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(proposed)}</td>
        </tr>
      </table>
      """
    if reference_name == "both":
      html = f"""
      <table style="border-collapse:collapse;">
        <tr>
          <th style="padding:6px;border:1px solid #ddd;">Case</th>
          <th style="padding:6px;border:1px solid #ddd;">Reference</th>
          <th style="padding:6px;border:1px solid #ddd;">GT</th>
          <th style="padding:6px;border:1px solid #ddd;">Baseline</th>
          <th style="padding:6px;border:1px solid #ddd;">Proposed</th>
        </tr>
        <tr>
          <td style="padding:6px;border:1px solid #ddd;">{case_name}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(reference)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(gt)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(baseline)}</td>
          <td style="padding:6px;border:1px solid #ddd;">{audio_tag(proposed)}</td>
        </tr>
      </table>
      """
    display(HTML(html))

In [68]:
## Instead of displaying wavs in order, we group them by the source and target emotion
## That is group them by direction

In [8]:
direction_list = [
    "Neutral --> Angry",
    "Neutral --> Happy",
    "Neutral --> Sad",
    "Neutral --> Surprise",
    "Angry --> Neutral",
    "Angry --> Happy",
    "Angry --> Sad",
    "Angry --> Surprise",
    "Happy --> Neutral",
    "Happy --> Angry",
    "Happy --> Sad",
    "Happy --> Surprise",
    "Sad --> Neutral",
    "Sad --> Angry",
    "Sad --> Happy",
    "Sad --> Surprise",
    "Surprise --> Neutral",
    "Surprise --> Angry",
    "Surprise --> Happy",
    "Surprise --> Sad",
]

In [26]:
# current_direction = "Neutral --> Angry"
# current_direction = "Neutral --> Happy"
# current_direction = "Neutral --> Sad"
# current_direction = "Neutral --> Surprise"

# current_direction = "Angry --> Neutral"
# current_direction = "Angry --> Happy"
current_direction = "Angry --> Sad"
# current_direction = "Angry --> Surprise"

# current_direction = "Happy --> Neutral"
# current_direction = "Happy --> Sad"
# current_direction = "Happy --> Angry"
# current_direction = "Happy --> Surprise"

# current_direction = "Sad --> Neutral"
# current_direction = "Sad --> Angry"
# current_direction = "Sad --> Surprise"
# current_direction = "Sad --> Happy"

# current_direction = "Surprise --> Neutral"
# current_direction = "Surprise --> Sad"
# current_direction = "Surprise --> Happy"
# current_direction = "Surprise --> Angry"

selected_wavs = wav_group[current_direction]
# wavs = wav_group[current_direction]

In [12]:
# selected_wavs = [
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0016_000335_Neutral_to_0016_000659_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0011_000326_Neutral_to_0011_000661_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0016_000333_Neutral_to_0016_000660_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_000346_Neutral_to_0019_000663_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_000321_Neutral_to_0019_000665_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_000333_Neutral_to_0019_000661_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_000322_Neutral_to_0019_000662_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0013_000332_Neutral_to_0013_000667_Angry.wav",
# ]
# selected_wavs = [
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0013_001729_Surprise_to_0013_000651_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0019_001735_Surprise_to_0019_000652_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0016_001721_Surprise_to_0016_000651_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0016_001739_Surprise_to_0016_000650_Angry.wav",
#     "./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0011_001725_Surprise_to_0011_000653_Angry.wav",
# ]

In [16]:
# for wav in selected_wavs:
#     reference_wav = get_reference_wav(wav)
#     baseline_wav = get_baseline_wav(wav)
#     gt_wav = get_gt_wav(wav)
#     proposed_wav = wav
#     wav_name = wav.split("/")[-1].split(".")[0]
#     parts = wav_name.split("_to_")
#     source_emo = parts[0].split("_")[-1]
#     target_emo = parts[1].split("_")[-1]
#     case_name = f"{source_emo} ---> {target_emo}"
#     print("wav", proposed_wav)
#     show_triplet(
#         case_name, 
#         reference_wav, 
#         gt_wav,
#         baseline_wav, 
#         proposed_wav,
#         reference_name="Reference"
#     )


In [57]:
final_list = "./demo_chosen_list_half.txt"
selected_wavs = []
with open(final_list, "r") as f:
    lines = f.readlines()
    for line in lines:
        line = line.strip()
        if os.path.exists(line):
            selected_wavs.append(line)


In [ ]:
## What to do 
## 1. Randomly Mix them but keep the numbers.
## 2. Get the reference and the gt.
## 3. Randomly assign with A and B between the reference and the gt.

In [88]:
import random
import json
import shutil
# random.shuffle(selected_wavs)

In [ ]:

# ## Keep the numbers.
# record_json = {}
# for i in range(len(selected_wavs)):
#     key_id = str(i+1)
#     record_json[key_id] = {}
#     # record_json[key_id]["Quality"] = {}
#     # record_json[key_id]["Similarity"] = {}
#     reference_wav = get_reference_wav(selected_wavs[i])
#     # gt_wav = get_gt_wav(selected_wavs[i])
#     baseline_wav = get_baseline_wav(selected_wavs[i])
#     proposed_wav = selected_wavs[i]
#     record_json[key_id]["Ref"] = reference_wav
#     prob = random.random()
#     if prob < 0.5:
#         record_json[key_id]["A"] = proposed_wav
#         record_json[key_id]["B"] = baseline_wav
#     else:
#         record_json[key_id]["A"] = baseline_wav
#         record_json[key_id]["B"] = proposed_wav
# with open("record_json.json", "w") as f:
#     json.dump(record_json, f, indent=4)


In [27]:
for wav in selected_wavs:
    reference_wav = get_reference_wav(wav)
    baseline_wav = get_baseline_wav(wav)
    # gt_wav = get_gt_wav(wav)
    gt_wav = get_source_wav(wav)
    proposed_wav = wav
    wav_name = wav.split("/")[-1].split(".")[0]
    parts = wav_name.split("_to_")
    source_emo = parts[0].split("_")[-1]
    target_emo = parts[1].split("_")[-1]
    case_name = f"{source_emo} ---> {target_emo}"
    
    print("SRC", gt_wav)
    print("baseline", baseline_wav)
    print("proposed", proposed_wav)
    show_triplet(
        case_name, 
        reference_wav, 
        gt_wav,
        baseline_wav, 
        proposed_wav,
        reference_name="GT"
    )

SRC /home/lxj220018/GenVC-EMO/ESD/0011/Angry/0011_000691.wav
baseline /home/lxj220018/GenVC-EMO/Amphion/output_demo/vevovoice/0011_000691_Angry_to_0011_001357_Sad_vevovoice.wav
proposed ./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0011_000691_Angry_to_0011_001357_Sad.wav


Case,GT,Baseline,Proposed
Angry ---> Sad,,,


SRC /home/lxj220018/GenVC-EMO/ESD/0013/Angry/0013_000675.wav
baseline /home/lxj220018/GenVC-EMO/Amphion/output_demo/vevovoice/0013_000675_Angry_to_0013_001361_Sad_vevovoice.wav
proposed ./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0013_000675_Angry_to_0013_001361_Sad.wav


Case,GT,Baseline,Proposed
Angry ---> Sad,,,


SRC /home/lxj220018/GenVC-EMO/ESD/0013/Angry/0013_000684.wav
baseline /home/lxj220018/GenVC-EMO/Amphion/output_demo/vevovoice/0013_000684_Angry_to_0013_001356_Sad_vevovoice.wav
proposed ./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0013_000684_Angry_to_0013_001356_Sad.wav


...

Case,GT,Baseline,Proposed
Angry ---> Sad,,,


SRC /home/lxj220018/GenVC-EMO/ESD/0016/Angry/0016_000687.wav
baseline /home/lxj220018/GenVC-EMO/Amphion/output_demo/vevovoice/0016_000687_Angry_to_0016_001368_Sad_vevovoice.wav
proposed ./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0016_000687_Angry_to_0016_001368_Sad.wav


Case,GT,Baseline,Proposed
Angry ---> Sad,,,


SRC /home/lxj220018/GenVC-EMO/ESD/0011/Angry/0011_000673.wav
baseline /home/lxj220018/GenVC-EMO/Amphion/output_demo/vevovoice/0011_000673_Angry_to_0011_001350_Sad_vevovoice.wav
proposed ./demo_test_deep_prefix_epoch-0012_step-0047000_loss-0.115780/0011_000673_Angry_to_0011_001350_Sad.wav


Case,GT,Baseline,Proposed
Angry ---> Sad,,,


In [86]:
with open("record_json.json", "r") as f:
    records = json.load(f)

In [ ]:
## According to the records, get a copy of those wavs and put them in the same folder.
listener_test_path = "./listener_test"
os.makedirs(listener_test_path, exist_ok=True)
for key_id, record in records.items():
    ref_wav = record["Ref"]
    ref_emo = ref_wav.split("/")[-2]
    # print(ref_emo)
    a_wav = record["A"]
    b_wav = record["B"]
    shutil.copy(ref_wav, os.path.join(listener_test_path, f"{key_id}_ref_{ref_emo}.wav"))
    shutil.copy(a_wav, os.path.join(listener_test_path, f"{key_id}_A.wav"))
    shutil.copy(b_wav, os.path.join(listener_test_path, f"{key_id}_B.wav"))

Happy
Sad
Surprise
Happy
Sad
Angry
Neutral
Happy
Happy
Happy
Sad
Surprise
Surprise
Sad
Neutral
Sad
Neutral
Sad
Sad
Surprise
Surprise
Surprise
Angry
Surprise
Neutral
Angry
Sad
Angry
Neutral
Happy
Neutral
Angry
Happy
Angry
Angry
Neutral
Neutral
Happy
Surprise
Angry
